In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import models, transforms
import torch.optim as optim
from torch.optim import lr_scheduler
import time
import os
import copy
import cv2

from scipy.stats import mode
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm import tqdm
from google.colab.patches import cv2_imshow
from PIL import Image

ModuleNotFoundError: No module named 'pandas'

In [ ]:
import os
import cv2
import math

from pathlib import Path
from torchvision import transforms

def create_patches(image_dir, patch_size, target_dir):
    target_dir_path = Path(target_dir)
    target_imgs_path, target_masks_path = Path(target_dir+'/'+'imgs/'), Path(target_dir+'/'+'masks/')
    #target_encoded_path = Path(target_dir+'/'+'encoded_masks/')
    target_dir_path.mkdir(parents=True, exist_ok=True)
    target_imgs_path.mkdir(parents=True, exist_ok=True)
    target_masks_path.mkdir(parents=True, exist_ok=True)
    #target_encoded_path.mkdir(parents=True, exist_ok=True)

    images_index, masks_index = 0, 0

    for path, _, _ in sorted(os.walk(image_dir)):
        dirname = path.split(os.path.sep)[-1]
        if dirname == 'images':
            images = sorted(os.listdir(path))
            for _, image_name in enumerate(images):
                if image_name.endswith(".jpg"):
                    #image = cv2.cvtColor(cv2.imread(path+"/"+image_name), cv2.COLOR_BGR2RGB)
                    image = cv2.imread(path+"/"+image_name)
                    size_X, size_Y = math.ceil(image.shape[1]/patch_size), math.ceil(image.shape[0]/patch_size)
                    pad_X, pad_Y = (patch_size * size_X - image.shape[1]) / (size_X - 1), (patch_size * size_Y - image.shape[0]) / (size_Y - 1)
                    image = Image.fromarray(image)
                    top = 0
                    for _ in range(size_Y):
                        left = 0
                        for _ in range(size_X):
                            crop_image = transforms.functional.crop(image, top, left, patch_size, patch_size)
                            crop_image = np.array(crop_image)
                            cv2.imwrite(f"{target_imgs_path}/image"+str(images_index).zfill(4)+".jpg", crop_image)
                            images_index += 1
                            left = left + patch_size - pad_X
                        top = top + patch_size - pad_Y
        
        if dirname == 'masks':
            images = sorted(os.listdir(path))
            for _, image_name in enumerate(images):
                if image_name.endswith(".png"):
                    #image = cv2.cvtColor(cv2.imread(path+"/"+image_name), cv2.COLOR_BGR2RGB)
                    image = cv2.imread(path+"/"+image_name)
                    size_X, size_Y = math.ceil(image.shape[1]/patch_size), math.ceil(image.shape[0]/patch_size)
                    pad_X, pad_Y = (patch_size * size_X - image.shape[1]) / (size_X - 1), (patch_size * size_Y - image.shape[0]) / (size_Y - 1)
                    image = Image.fromarray(image)
                    top = 0
                    for _ in range(size_Y):
                        left = 0
                        for _ in range(size_X):
                            crop_image = transforms.functional.crop(image, top, left, patch_size, patch_size)
                            crop_image = np.array(crop_image)
                            cv2.imwrite(f"{target_masks_path}/image"+str(masks_index).zfill(4)+".png", crop_image)
                            #encoded_image = one_hot_encode_masks(crop_image, 6)
                            #cv2.imwrite(f"{target_encoded_path}/image"+str(masks_index).zfill(4)+".png", encoded_image)
                            masks_index += 1
                            left = left + patch_size - pad_X
                        top = top + patch_size - pad_Y